> **기록 표시 (2026-11-05):** 이 노트북은 항목 2 추출 당시의 검수 대기 기록이다. 추천안 승인으로 행 2개가 승인 파일에 등록되었고, 노트북 25에서 Chunk 2·행동 2·관계 10개가 적재되었다. 현재 상태는 `일정관리_prevention_legal.md`를 본다.


# 24. 93쪽 항목 2 추출 (검수 대기, 블록 b05·b06)

- 목적: 항목 2(진단 받은 경우 등교·등원 금지, 익일 등교 안내)를 v2 추출 프롬프트로 추출한다. 이 결과는 Q02·Q03 근거가 된다.
- 방법: 04 v2 노트북의 스키마와 프롬프트를 그대로 가져온다.
- 저장: `steps_pending_item2.jsonl`에 `review_status = unreviewed`로만 쓴다. 승인 파일은 건드리지 않는다.
- 승인은 사용자가 한다.

In [1]:
import json, pathlib, re
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

# 04 v2 노트북의 스키마(셀 4)와 프롬프트(셀 6)를 그대로 가져온다
src = json.load(open("04_추출프롬프트_v2.ipynb", encoding="utf-8"))
ns = {}
for idx in (4, 6):
    exec("".join(src["cells"][idx]["source"]), ns)

BLOCK_IDS = ["kdca_measles_2026:p0093:b05", "kdca_measles_2026:p0093:b06"]
blocks = {}
for l in open("../../data/interim/prevention_legal/blocks.jsonl", encoding="utf-8"):
    if l.strip():
        b = json.loads(l)
        blocks[b["chunk_id"]] = b

load_dotenv(find_dotenv(usecwd=True), override=False)   # API 키는 .env 에서 읽는다 (값은 출력하지 않음)
model = init_chat_model("openai:gpt-6-luna", reasoning_effort="none")
extractor = model.with_structured_output(ns["ExtractionResult"])

def compact(s):
    return re.sub(r"[\s*\[\]]", "", s)

rows = []
for bid in BLOCK_IDS:
    block = blocks[bid]
    section = block.get("section") or "(소제목 없음)"
    human = ("문서: " + block["document_id"] + " / " + str(block["pdf_page"]) + "쪽 / 블록 " + block["chunk_id"]
             + "\n소제목(문맥): " + section + "\n\n원문:\n" + block["text"])
    out = extractor.invoke([("system", ns["PROMPT_V2"]), ("human", human)])
    print(bid, "행동", len(out.actions))
    for n, a in enumerate(out.actions, start=1):
        d = a.model_dump()
        quote_ok = compact(d["evidence_quote"]) in compact(block["text"])
        rows.append({
            "run_id": "pending_item2", "chunk_id": bid, "action_no": str(n), "status": "ok",
            "action_text": d["action_text"], "stage": d["stage"], "scope": d["scope"], "modality": d["modality"],
            "when_text": d["when_text"], "timing_text": d["timing_text"], "role_name": d["role_name"],
            "basis": d["basis"], "evidence_quote": d["evidence_quote"],
            "quote_check": "블록 일치" if quote_ok else "블록 불일치",
            "context_section": section, "review_status": "unreviewed",
            "model": "gpt-6-luna", "prompt_version": "v2", "review_note": "검수 대기 (사용자 승인 전)",
        })
        print("  ", d["action_text"], "|", rows[-1]["quote_check"], "|", d["basis"], "|", d["modality"], "|", d["when_text"])

out_path = pathlib.Path("../../data/interim/prevention_legal/steps_pending_item2.jsonl")
out_path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
print("저장:", out_path.name, len(rows), "행 (검수 대기)")


스키마 준비 완료
648 자 프롬프트 v2 준비


kdca_measles_2026:p0093:b05 행동 2
   홍역(의심) 진단을 받은 경우 환자에게 전염기간 동안 학교에 등교하지 않도록 안내한다. | 블록 일치 | EXPLICIT | UNSPECIFIED | 의료기관 진료 결과, 홍역(의심) 진단을 받은 경우
   홍역(의심) 진단을 받은 경우 환자에게 전염기간 동안 학원에 등원하지 않도록 안내한다. | 블록 일치 | EXPLICIT | UNSPECIFIED | 의료기관 진료 결과, 홍역(의심) 진단을 받은 경우


kdca_measles_2026:p0093:b06 행동 1
   실험실 검사 결과 환자 아님으로 확인되면 익일 등교를 안내한다. | 블록 일치 | EXAMPLE | UNSPECIFIED | 실험실 검사 결과 환자 아님으로 확인 시
저장: steps_pending_item2.jsonl 3 행 (검수 대기)
